In [2]:
!pip install -q kagglehub ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.7 MB/s eta 0:00:00


In [3]:
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
import kagglehub, json, os, shutil, glob, random
from pathlib import Path
from collections import defaultdict

# Download
path = kagglehub.dataset_download("kneroma/tacotrashdataset")
print("Downloaded to:", path)

# Convert COCO -> YOLO format
TACO_DATA = f"{path}/data"
ANN_PATH = f"{TACO_DATA}/annotations.json"

with open(ANN_PATH) as f:
    coco = json.load(f)

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

out_images_dir = Path("dataset_flat/images")
out_labels_dir = Path("dataset_flat/labels")
out_images_dir.mkdir(parents=True, exist_ok=True)
out_labels_dir.mkdir(parents=True, exist_ok=True)

copied, skipped = 0, 0
for img_id, img in images_by_id.items():
    file_name = img["file_name"]
    src_path = os.path.join(TACO_DATA, file_name)
    if not os.path.exists(src_path):
        skipped += 1
        continue
    batch_part, fname = file_name.split("/")
    unique_stem = f"{batch_part}_{Path(fname).stem}"
    shutil.copy(src_path, out_images_dir / f"{unique_stem}{Path(fname).suffix}")
    w, h = img["width"], img["height"]
    lines = []
    for ann in anns_by_image.get(img_id, []):
        x, y, bw, bh = ann["bbox"]
        xc, yc = (x + bw / 2) / w, (y + bh / 2) / h
        nw, nh = bw / w, bh / h
        lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
    with open(out_labels_dir / f"{unique_stem}.txt", "w") as lf:
        lf.write("\n".join(lines))
    copied += 1

print(f"Copied {copied} images, skipped {skipped}")

# Train/val split
random.seed(42)
label_files = glob.glob("dataset_flat/labels/*.txt")
stems = [os.path.splitext(os.path.basename(f))[0] for f in label_files]
random.shuffle(stems)
split_idx = int(len(stems) * 0.8)
train_stems, val_stems = stems[:split_idx], stems[split_idx:]

for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok=True)
    os.makedirs(f"dataset/labels/{split}", exist_ok=True)

def place(stem_list, split):
    for stem in stem_list:
        matches = glob.glob(f"dataset_flat/images/{stem}.*")
        if not matches:
            continue
        img_src = matches[0]
        img_ext = os.path.splitext(img_src)[1]
        shutil.copy(img_src, f"dataset/images/{split}/{stem}{img_ext}")
        shutil.copy(f"dataset_flat/labels/{stem}.txt", f"dataset/labels/{split}/{stem}.txt")

place(train_stems, "train")
place(val_stems, "val")
print(f"Train: {len(train_stems)}, Val: {len(val_stems)}")

100%|██████████| 2.79G/2.79G [00:45<00:00, 65.6MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/kneroma/tacotrashdataset/versions/3
Copied 1500 images, skipped 0
Train: 1200, Val: 300


In [5]:
%%writefile data.yaml
path: /content/dataset
train: images/train
val: images/val
names:
  0: garbage

Writing data.yaml


In [6]:
import torch
print("GPU available:", torch.cuda.is_available())

from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='data.yaml', epochs=15, imgsz=640, batch=16)

GPU available: False
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d50ac1cd3d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
from google.colab import files
files.download('runs/detect/train/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>